In [1]:
# -*- coding: utf-8 -*-
"""
SIC 裁剪 Sentinel-1（支持 GEE 导出 ×100→int16 的还原；Bremen SIC 0–100 + 缓冲带）
输出两份可选文件：
  1) *_float32.tif  : 还原后的浮点（推荐用于后续波段计算、阈值/聚类）
  2) *_int16x100.tif: 可选的重压缩存档（×100→int16），节省空间
"""

import os
from pathlib import Path
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject
import matplotlib.pyplot as plt

# ========== 路径 ==========
S1_STACK_PATH = r"F:\NWP\sentinel1 gee\2023\S1A_EW_GRDM_1SDH_20230412T121730_20230412T121835_048063_05C70C_40A9_EW_HH_HV_angle_int16x100_87caa6ee.tif"  # 单文件多波段
# 或者分别提供（留空代表不用）
S1_HH_PATH    = r""
S1_HV_PATH    = r""
S1_ANGLE_PATH = r""

SIC_PATH = r"E:\NWP\CS2_S1_matched\SIC\n6250\asi-AMSR2-n6250-20230412-v5.tif" # Bremen SIC，0–100

OUT_DIR = r"F:\NWP\sentinel1 maskbySIC\2023"
os.makedirs(OUT_DIR, exist_ok=True)

# ========== GEE 压缩还原 & 保存选项 ==========
# 指定每个波段的还原因子（×factor）。GEE 压缩通常是把线性/或dB值×100存成int16。
# 若你的 ANGLE 也压缩过，可把 1.0 改为 0.01。
PER_BAND_SCALE = {"HH": 0.01, "HV": 0.01, "ANGLE": 1.0}
SAVE_INT16_X100 = True   # 是否同时保存一个重新×100压缩的 int16 版本
INT16_NODATA    = -32768 # int16 nodata

# ========== SIC 缓冲带 & 掩膜模式 ==========
# Bremen SIC 范围 0–100。常见阈值：水≤10%、冰≥15%，中间 10–15% 作为缓冲带剔除
SIC_WATER_MAX = 10.0   # ≤此值判作水
SIC_ICE_MIN   = 15.0   # ≥此值判作冰
ICE_MAX = 100.0  # SIC 最大值
# 掩膜模式：保留何种区域
#   "ice"   : 仅保留海冰 (SIC≥ICE_MIN)
#   "water" : 仅保留开阔水 (SIC≤WATER_MAX)
MASK_MODE = "ice"

# 重采样方法（把 SIC 对齐到 S1）
SIC_RESAMPLING = Resampling.bilinear

# 是否输出 quicklook
GENERATE_QUICKLOOK = True


In [14]:

# ----------------- 工具函数 -----------------
def read_s1_bands(stack_path, hh_path="", hv_path="", angle_path=""):
    band_names = []
    if Path(stack_path).exists():
        with rasterio.open(stack_path) as src:
            profile = src.profile.copy()
            data = src.read()  # (B,H,W)
            # 波段名
            descs = []
            try:
                descs = [src.descriptions[i] for i in range(src.count)]
            except Exception:
                pass
            if descs and all(d is not None for d in descs):
                band_names = [d.upper() for d in descs]
            else:
                # 猜测顺序
                band_names = []
                if src.count >= 1: band_names.append("HH")
                if src.count >= 2: band_names.append("HV")
                if src.count >= 3: band_names.append("ANGLE")
            return data, profile, band_names

    arrays = []
    base_profile = None
    if hh_path and Path(hh_path).exists():
        with rasterio.open(hh_path) as src:
            arrays.append(src.read(1)[None, ...]); base_profile = base_profile or src.profile.copy(); band_names.append("HH")
    if hv_path and Path(hv_path).exists():
        with rasterio.open(hv_path) as src:
            arrays.append(src.read(1)[None, ...]); base_profile = base_profile or src.profile.copy(); band_names.append("HV")
    if angle_path and Path(angle_path).exists():
        with rasterio.open(angle_path) as src:
            arrays.append(src.read(1)[None, ...]); base_profile = base_profile or src.profile.copy(); band_names.append("ANGLE")

    if not arrays:
        raise FileNotFoundError("未找到 Sentinel-1 输入影像。")

    data_stack = np.concatenate(arrays, axis=0)
    return data_stack, base_profile, band_names

# 1) 由“原始 int16（还原前）”识别 S1 外部零值
def build_s1_zero_mask_int16(s1_data_raw, band_names):
    """
    Returns: valid_mask (H, W) -> True 表示 S1 内部有效像元
    规则：只要 HH==0 或 HV==0，就视为无效（外部空值）；否则有效。
    （用原始 int16 判断，避免把真实的 0.00 浮点当外部值）
    """
    name_upper = [n.upper() for n in band_names]
    hh_idx = name_upper.index("HH") if "HH" in name_upper else None
    hv_idx = name_upper.index("HV") if "HV" in name_upper else None

    H, W = s1_data_raw.shape[1], s1_data_raw.shape[2]
    valid = np.ones((H, W), dtype=bool)

    if hh_idx is not None:
        valid &= (s1_data_raw[hh_idx] != 0)
    if hv_idx is not None:
        valid &= (s1_data_raw[hv_idx] != 0)

    return valid  # True=有效，False=无效（外部零值）

def to_float32_restore_scale(data_stack, band_names, per_band_scale):
    """按波段名把 ×100→int16 的数据还原为 float32"""
    data_stack = data_stack.astype(np.float32)
    for i, name in enumerate([n.upper() for n in band_names]):
        scale = per_band_scale.get(name, 1.0)
        if scale != 1.0:
            data_stack[i] *= scale
    return data_stack

def align_sic_to_s1(sic_path, s1_profile, resampling=SIC_RESAMPLING):
    with rasterio.open(sic_path) as src:
        src_nodata = src.nodata
        dst = np.full((s1_profile["height"], s1_profile["width"]), np.nan, dtype=np.float32)
        reproject(
            source=rasterio.band(src, 1),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src_nodata,
            dst_transform=s1_profile["transform"],
            dst_crs=s1_profile["crs"],
            dst_nodata=np.nan,
            resampling=resampling,
        )
    return dst

# 2) SIC 掩膜（带有效域、缓冲带、上界）
def make_keep_mask_from_sic_with_bounds(
    sic_arr,
    mode,
    water_max,
    ice_min,
    ice_max
):
    """
    - 仅在 sic ∈ [0,100] 的像元上考虑分类，其它一律无效（False）。
    - 缓冲带 (water_max, ice_min) 统一剔除（False）。
    - mode="ice"  : keep = (ice_min ≤ sic ≤ ice_max)
      mode="water": keep = (sic ≤ water_max)
    """
    sic_valid = np.isfinite(sic_arr) & (sic_arr >= 0.0) & (sic_arr <= 100.0)

    is_water = sic_arr <= water_max
    is_ice   = (sic_arr >= ice_min) & (sic_arr <= ice_max)

    if mode == "ice":
        keep = is_ice
    elif mode == "water":
        keep = is_water
    else:
        raise ValueError("MASK_MODE 仅支持 'ice' 或 'water'")

    # 去掉缓冲带
    in_buffer = (sic_arr > water_max) & (sic_arr < ice_min)

    keep = keep & sic_valid & (~in_buffer)
    keep = np.where(np.isfinite(sic_arr), keep, False)
    return keep

def save_float32(out_path, data_stack, base_profile):
    prof = base_profile.copy()
    prof.update(count=data_stack.shape[0], dtype="float32", nodata=np.nan, compress="DEFLATE", tiled=True, BIGTIFF="IF_SAFER")
    with rasterio.open(out_path, "w", **prof) as dst:
        dst.write(data_stack)

def save_int16_x100(out_path, data_stack_float, band_names, nodata_val=INT16_NODATA):
    """把 float 再×100 → int16 存盘（仅对 HH/HV；ANGLE 原样或另行处理）"""
    out = []
    for i, name in enumerate([n.upper() for n in band_names]):
        arr = data_stack_float[i].copy()
        if name in ("HH", "HV"):           # 只对散射系数做 ×100
            arr = np.where(np.isfinite(arr), arr * 100.0, np.nan)
        # 量化到 int16
        arr = np.where(np.isfinite(arr), np.clip(np.round(arr), -32767, 32767), nodata_val).astype(np.int16)
        out.append(arr)
    out = np.stack(out, axis=0)

    prof = base_profile.copy()
    prof.update(count=out.shape[0], dtype="int16", nodata=nodata_val, compress="DEFLATE", tiled=True, BIGTIFF="IF_SAFER")
    with rasterio.open(out_path, "w", **prof) as dst:
        dst.write(out)


In [15]:
s1_data_raw, s1_profile, band_names = read_s1_bands(S1_STACK_PATH, S1_HH_PATH, S1_HV_PATH, S1_ANGLE_PATH)
H, W = s1_profile["height"], s1_profile["width"]

In [16]:
print(f"S1 loaded: shape={s1_data_raw.shape}, bands={band_names}, size={H}x{W}")

S1 loaded: shape=(3, 13440, 13451), bands=['HH', 'HV', 'ANGLE'], size=13440x13451


In [17]:
print(s1_profile)

{'driver': 'GTiff', 'dtype': 'int16', 'nodata': None, 'width': 13451, 'height': 13440, 'count': 3, 'crs': CRS.from_epsg(32618), 'transform': Affine(40.0, 0.0, 282895.4673565922,
       0.0, -40.0, 8994766.077283295), 'blockxsize': 256, 'blockysize': 256, 'tiled': True, 'compress': 'lzw', 'interleave': 'pixel'}


In [18]:
# 2) 基于“原始 int16”构建 S1 外部零值掩膜（True=有效）
s1_valid_mask = build_s1_zero_mask_int16(s1_data_raw, band_names)

In [19]:
s1_float = to_float32_restore_scale(s1_data_raw, band_names, PER_BAND_SCALE)

In [20]:
sic_aligned = align_sic_to_s1(SIC_PATH, s1_profile, resampling=SIC_RESAMPLING)

In [21]:
keep_mask_sic = make_keep_mask_from_sic_with_bounds(
    sic_aligned, MASK_MODE, SIC_WATER_MAX, SIC_ICE_MIN, ICE_MAX
)

In [22]:
# 6) 组合掩膜：S1 有效 & SIC 保留
final_keep = s1_valid_mask & keep_mask_sic

In [23]:
# 7) 应用掩膜（float 输出中用 NaN）
s1_masked_float = s1_float.copy()
for b in range(s1_masked_float.shape[0]):
    band = s1_masked_float[b]
    band[~final_keep] = np.nan
    s1_masked_float[b] = band

In [24]:
# 8) 保存
base_name = Path(S1_STACK_PATH if Path(S1_STACK_PATH).exists() else S1_HH_PATH).stem
out_float = os.path.join(OUT_DIR, f"{base_name}_masked_bySIC_{MASK_MODE}_w{SIC_WATER_MAX}_i{SIC_ICE_MIN}_imax{int(ICE_MAX)}_float32.tif")
save_float32(out_float, s1_masked_float, s1_profile)
print(f"✅ Saved float32: {out_float}")

✅ Saved float32: F:\NWP\sentinel1 maskbySIC\S1A_EW_GRDM_1SDH_20230412T121730_20230412T121835_048063_05C70C_40A9_EW_HH_HV_angle_int16x100_87caa6ee_masked_bySIC_ice_w10.0_i15.0_imax100_float32.tif


In [ ]:

# 9) Quicklook（示例：HH）
if GENERATE_QUICKLOOK:
    try:
        hh_idx = 0
        for i, n in enumerate([n.upper() for n in band_names]):
            if n in ("HH", "SIGMA0_HH"):
                hh_idx = i; break
        hh = s1_masked_float[hh_idx]
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        im0 = axes[0].imshow(hh, vmin=np.nanpercentile(hh, 2), vmax=np.nanpercentile(hh, 98))
        axes[0].set_title(f"S1 {band_names[hh_idx]} (masked)")
        axes[0].axis("off"); fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

        im1 = axes[1].imshow(sic_aligned); axes[1].contour(final_keep.astype(np.uint8), levels=[0.5], linewidths=0.8)
        axes[1].set_title("Bremen SIC (0–100), contour=kept"); axes[1].axis("off"); fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

        quick_png = os.path.join(OUT_DIR, f"{base_name}_quicklook_{MASK_MODE}_w{SIC_WATER_MAX}_i{SIC_ICE_MIN}_imax{int(ICE_MAX)}.png")
        plt.tight_layout(); plt.savefig(quick_png, dpi=200); plt.close(fig)
        print(f"🖼️ Saved quicklook: {quick_png}")
    except Exception as e:
        print("Quicklook failed:", e)

🖼️ Saved quicklook: F:\NWP\sentinel1 maskbySIC\S1A_EW_GRDM_1SDH_20230412T121730_20230412T121835_048063_05C70C_40A9_EW_HH_HV_angle_int16x100_87caa6ee_quicklook_ice_w10.0_i15.0_imax100.png


In [ ]:
# ----------------- 主流程（替换原先步骤 3-5 部分） -----------------
def main():
    # 1) 读取 S1（原始 int16×100）
    s1_data_raw, s1_profile, band_names = read_s1_bands(S1_STACK_PATH, S1_HH_PATH, S1_HV_PATH, S1_ANGLE_PATH)
    H, W = s1_profile["height"], s1_profile["width"]
    print(f"S1 loaded: shape={s1_data_raw.shape}, bands={band_names}, size={H}x{W}")

    # 2) 基于“原始 int16”构建 S1 外部零值掩膜（True=有效）
    s1_valid_mask = build_s1_zero_mask_int16(s1_data_raw, band_names)

    # 3) 还原到 float32（×0.01 等）
    s1_float = to_float32_restore_scale(s1_data_raw, band_names, PER_BAND_SCALE)

    # 4) 对齐 SIC（Bremen 0–100）
    sic_aligned = align_sic_to_s1(SIC_PATH, s1_profile, resampling=SIC_RESAMPLING)

    # 5) 生成 SIC keep_mask（带有效域、缓冲带、上界）
    # 你可以在顶部参数区新增 ICE_MAX（默认 100）
    ICE_MAX = 100.0
    keep_mask_sic = make_keep_mask_from_sic_with_bounds(
        sic_aligned, MASK_MODE, SIC_WATER_MAX, SIC_ICE_MIN, ICE_MAX
    )

    # 6) 组合掩膜：S1 有效 & SIC 保留
    final_keep = s1_valid_mask & keep_mask_sic

    # 7) 应用掩膜（float 输出中用 NaN）
    s1_masked_float = s1_float.copy()
    for b in range(s1_masked_float.shape[0]):
        band = s1_masked_float[b]
        band[~final_keep] = np.nan
        s1_masked_float[b] = band

    # 8) 保存
    base_name = Path(S1_STACK_PATH if Path(S1_STACK_PATH).exists() else S1_HH_PATH).stem
    out_float = os.path.join(OUT_DIR, f"{base_name}_masked_bySIC_{MASK_MODE}_w{SIC_WATER_MAX}_i{SIC_ICE_MIN}_imax{int(ICE_MAX)}_float32.tif")
    save_float32(out_float, s1_masked_float, s1_profile)
    print(f"✅ Saved float32: {out_float}")

    if SAVE_INT16_X100:
        out_i16 = os.path.join(OUT_DIR, f"{base_name}_masked_bySIC_{MASK_MODE}_w{SIC_WATER_MAX}_i{SIC_ICE_MIN}_imax{int(ICE_MAX)}_int16x100.tif")
        save_int16_x100(out_i16, s1_masked_float, band_names)
        print(f"✅ Saved int16×100: {out_i16}")

    # 9) Quicklook（示例：HH）
    if GENERATE_QUICKLOOK:
        try:
            hh_idx = 0
            for i, n in enumerate([n.upper() for n in band_names]):
                if n in ("HH", "SIGMA0_HH"):
                    hh_idx = i; break
            hh = s1_masked_float[hh_idx]
            fig, axes = plt.subplots(1, 2, figsize=(12, 5))
            im0 = axes[0].imshow(hh, vmin=np.nanpercentile(hh, 2), vmax=np.nanpercentile(hh, 98))
            axes[0].set_title(f"S1 {band_names[hh_idx]} (masked)")
            axes[0].axis("off"); fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

            im1 = axes[1].imshow(sic_aligned); axes[1].contour(final_keep.astype(np.uint8), levels=[0.5], linewidths=0.8)
            axes[1].set_title("Bremen SIC (0–100), contour=kept"); axes[1].axis("off"); fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

            quick_png = os.path.join(OUT_DIR, f"{base_name}_quicklook_{MASK_MODE}_w{SIC_WATER_MAX}_i{SIC_ICE_MIN}_imax{int(ICE_MAX)}.png")
            plt.tight_layout(); plt.savefig(quick_png, dpi=200); plt.close(fig)
            print(f"🖼️ Saved quicklook: {quick_png}")
        except Exception as e:
            print("Quicklook failed:", e)

if __name__ == "__main__":
    main()


In [2]:

import os
import re
from pathlib import Path
from datetime import datetime, timedelta
import glob
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject
import matplotlib.pyplot as plt

# ===================== 路径 & 基本设置 =====================
S1_FOLDER   = r"F:\NWP\sentinel1 gee\2023"
SIC_FOLDER  = r"E:\NWP\CS2_S1_matched\SIC\n6250"
OUT_DIR     = r"F:\NWP\sentinel1 maskbySIC\2023"
os.makedirs(OUT_DIR, exist_ok=True)

# 匹配 S1 文件的模式（可按需调整）
S1_GLOB_PATTERNS = [
    "*.tif", "*.tiff"
]


In [3]:
# ===================== 辅助函数 =====================
def list_s1_files(folder, patterns):
    files = []
    for pat in patterns:
        files.extend(glob.glob(os.path.join(folder, pat)))
    files = [f for f in files if Path(f).is_file()]
    files.sort()
    return files

def extract_date_from_name(name: str):
    """
    从文件名中提取日期，返回 datetime.date；提取失败返回 None
    策略：
      1) 先找连续8位数字形如 YYYYMMDD
      2) 再尝试 YYYY[-/_]?MM[-/_]?DD
      3) 再尝试 YYYYMM（默认日=15作为中位；不建议，但保底）
    """
    base = Path(name).stem

    # 1) 8位 YYYYMMDD
    m = re.search(r"(20\d{2})(\d{2})(\d{2})", base)
    if m:
        y, mo, d = map(int, m.groups())
        try:
            return datetime(y, mo, d).date()
        except ValueError:
            pass

    # 2) YYYY[-/_]?MM[-/_]?DD
    m = re.search(r"(20\d{2})[-_/]?(\d{2})[-_/]?(\d{2})", base)
    if m:
        y, mo, d = map(int, m.groups())
        try:
            return datetime(y, mo, d).date()
        except ValueError:
            pass

    # 3) YYYYMM（fallback，日=15）
    m = re.search(r"(20\d{2})(\d{2})", base)
    if m:
        y, mo = map(int, m.groups())
        try:
            return datetime(y, mo, 15).date()
        except ValueError:
            pass

    return None

def index_sic_by_date(sic_folder):
    """
    建立 {date: [paths]} 索引；一个日期可能有多条（早/晚轨、不同版本）
    """
    idx = {}
    for tif in glob.glob(os.path.join(sic_folder, "*.tif*")):
        d = extract_date_from_name(tif)
        if d is None:
            continue
        idx.setdefault(d, []).append(tif)
    # 每日多文件时，默认按文件名排序（你也可以自定义优先规则）
    for k in idx:
        idx[k].sort()
    return idx

def pick_sic_for_date(sic_index, target_date):
    """
    选择与 target_date 最匹配的 SIC：
      - 优先同日；
      - 若 SEARCH_NEAREST_DAYS > 0，允许 ±N 天内就近匹配（返回最近的那天）。
    返回：path 或 None
    """
    if target_date in sic_index:
        return sic_index[target_date][0]
    if SEARCH_NEAREST_DAYS > 0:
        best = None
        best_dt = None
        for dt, paths in sic_index.items():
            diff = abs((dt - target_date).days)
            if diff <= SEARCH_NEAREST_DAYS:
                if best is None or diff < best:
                    best = diff
                    best_dt = dt
        if best_dt is not None:
            return sic_index[best_dt][0]
    return None

In [6]:
sic_index = index_sic_by_date(SIC_FOLDER)
if not sic_index:
    print(f"[WARN] No SIC files indexed under: {SIC_FOLDER}")

s1_files = list_s1_files(S1_FOLDER, S1_GLOB_PATTERNS)
if not s1_files:
    print(f"[ERROR] No Sentinel-1 files found under: {S1_FOLDER}")

print(f"Found {len(s1_files)} S1 files.")
done = 0
for s1_path in s1_files:
    s1_name = Path(s1_path).name
    s1_date = extract_date_from_name(s1_name)

    if s1_date is None:
        print(f"[SKIP] {s1_name}: no date found in filename.")
        continue

    sic_path = pick_sic_for_date(sic_index, s1_date)
    print('ok')
    if sic_path is None:
        print(f"[MISS] {s1_name}: no SIC found for date {s1_date} (±{SEARCH_NEAREST_DAYS} days).")
        continue

Found 250 S1 files.
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
